000

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
!unzip /content/drive/MyDrive/kaggle_datasets/home-data-for-ml-course.zip

Archive:  /content/drive/MyDrive/kaggle_datasets/home-data-for-ml-course.zip
  inflating: data_description.txt    
  inflating: sample_submission.csv   
  inflating: sample_submission.csv.gz  
  inflating: test.csv                
  inflating: test.csv.gz             
  inflating: train.csv               
  inflating: train.csv.gz            


In [ ]:
dataset = pd.read_csv('/content/train.csv', index_col='Id')

In [ ]:
dataset.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
Id,,,,,,,,,,,,,,,,,,,,,
1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [ ]:
dataset.select_dtypes('object').nunique().sort_values()

,0
Street,2
Alley,2
Utilities,2
CentralAir,2
MasVnrType,3
LandSlope,3
PavedDrive,3
GarageFinish,3
PoolQC,3
ExterQual,4


In [ ]:
processed = dataset.dropna(axis=1).select_dtypes(exclude='object')

In [ ]:
processed.head()

,MSSubClass,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
Id,,,,,,,,,,,,,,,,,,,,,
1,60,8450,7,5,2003,2003,706,0,150,856,...,0,61,0,0,0,0,0,2,2008,208500
2,20,9600,6,8,1976,1976,978,0,284,1262,...,298,0,0,0,0,0,0,5,2007,181500
3,60,11250,7,5,2001,2002,486,0,434,920,...,0,42,0,0,0,0,0,9,2008,223500
4,70,9550,7,5,1915,1970,216,0,540,756,...,0,35,272,0,0,0,0,2,2006,140000
5,60,14260,8,5,2000,2000,655,0,490,1145,...,192,84,0,0,0,0,0,12,2008,250000


In [ ]:
processed.shape

(1460, 34)

In [ ]:
corr_list = processed.corr()['SalePrice'].sort_values(ascending=False)
# print(corr_list)

In [ ]:
corr_features = list(corr_list[1:10].index)
print(corr_features)
kaggle_features = ['LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd']

['OverallQual', 'GrLivArea', 'GarageCars', 'GarageArea', 'TotalBsmtSF', '1stFlrSF', 'FullBath', 'TotRmsAbvGrd', 'YearBuilt']


In [ ]:
# X = processed[corr_features]
X = dataset[corr_features]
# X = dataset[kaggle_features]
y = dataset.SalePrice
# X = processed[kaggle_features]
# y = processed.SalePrice

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, random_state=1)

In [ ]:
dataset.shape

(1460, 80)

In [ ]:
X_train.shape

(1095, 9)

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

# model = keras.Sequential([
#     layers.Dense(512, activation='relu', input_shape=[9]),
#     layers.Dropout(0.3),
#     layers.BatchNormalization(),
#     layers.Dense(512, activation='relu'),
#     layers.Dropout(0.3),
#     layers.BatchNormalization(),
#     layers.Dense(512, activation='relu'),
#     layers.Dropout(0.3),
#     layers.BatchNormalization(),
#     layers.Dense(1),
# ])
model = keras.Sequential([
    layers.Dense(32, activation='relu', input_shape=[9]),
    layers.Dropout(0.3),
    layers.BatchNormalization(),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.BatchNormalization(),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.BatchNormalization(),
    layers.Dense(1),
])
# model = keras.Sequential([
#     layers.Dense(512, activation='relu', input_shape=[9]),
#     layers.Dense(512, activation='relu'),
#     layers.Dense(512, activation='relu'),
#     layers.Dense(1),
# ])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
from tensorflow.keras.optimizers import Adam

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.4),
    # optimizer='adam',
    loss='mae',
)

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=128,
    epochs=100,
    # batch_size=256,
    # epochs=10,
)

Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 33351.3984 - val_loss: 36980.3359
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 34367.7891 - val_loss: 31507.6387
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 34614.3398 - val_loss: 30749.7559
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 32624.7832 - val_loss: 37321.1406
Epoch 5/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 32161.6328 - val_loss: 33217.4258
Epoch 6/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 33029.7461 - val_loss: 36769.8047
Epoch 7/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 32244.8535 - val_loss: 34865.9492
Epoch 8/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 32675.1660 - val_loss: 32704.6523
Epoch 9/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 32216.7852 - val_loss: 35713.5508
Epoch 10/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 32660.4414 - val_loss: 33627.1250
Epoch 11/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 32224.3398 - val_los

In [ ]:
preds = model.predict(X_valid)

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step


In [ ]:
from sklearn.metrics import mean_absolute_error

In [ ]:
print(f'MAE: {mean_absolute_error(preds, y_valid)}')

MAE: 176695.234375
